In [ ]:
pip install -U transformers accelerate sentencepiece gradio torch

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
def build_prompt(context, question, level):
    return f"""
You are a senior university professor.

TASK:
Write a long, detailed, structured academic answer.
Minimum 600 words.
Use headings and paragraphs.
Maintain formal academic tone.
Explain concepts clearly.
Do not use random numbering.
Do not stop early.

Difficulty Level: {level}

CONTEXT:
<<<
{context}
>>>

QUESTION:
<<<
{question}
>>>

ANSWER:
"""

In [ ]:
def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=900,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
import gradio as gr

def academic_assistant(context, question, level):
    prompt = build_prompt(context, question, level)
    return generate_answer(prompt)

with gr.Blocks(title="Intelligent Academic Assistant") as demo:
    gr.Markdown("## 📘 Intelligent Academic Assistant (LLM Powered)")

    level = gr.Dropdown(
        ["School", "Undergraduate", "Postgraduate"],
        value="Undergraduate",
        label="Difficulty Level"
    )

    context = gr.Textbox(lines=8, label="Context")
    question = gr.Textbox(lines=3, label="Question")
    output = gr.Textbox(lines=15, label="Answer")

    gr.Button("Generate").click(
        academic_assistant,
        inputs=[context, question, level],
        outputs=output
    )

demo.launch()